# 02 - EDA (Plotly)

Szybki oglad obu datasetow przed modelowaniem. Pare wykreslow, ktore pomagaja
zrozumiec dane *i* uzasadniaja pozniejsze decyzje (np. czemu drzewa, a nie liniowa).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

laps = pd.read_parquet('../data/laps_clean.parquet')
dr = pd.read_parquet('../data/driver_race.parquet')
print('laps:', laps.shape, '| driver_race:', dr.shape)

## Rozklad czasow okrazen

In [ ]:
fig = px.histogram(laps, x='LapTime_s', nbins=80,
                   title='Rozklad czasow okrazen - sezon 2024 (po czyszczeniu)')
fig.show()

## Degradacja opon per mieszanka

To jest sedno regresji: widac nieliniowy wzrost czasu z wiekiem opony,
inny dla kazdej mieszanki -> liniowy model tego dobrze nie zlapie.

In [ ]:
# mediana czasu per (mieszanka, wiek opony), tylko dobrze obsadzone punkty
g = laps.groupby(['Compound', 'TyreLife'])['LapTime_s']
deg = g.agg(['median', 'count']).reset_index()
deg = deg[deg['count'] >= 5]
fig = px.line(deg, x='TyreLife', y='median', color='Compound', markers=True,
              title='Mediana czasu vs wiek opony per mieszanka')
fig.show()

## Efekt paliwa

Przez wyscig auto sie odchudza -> czasy spadaja niezaleznie od opon.
`LapNumber` to czesciowo lapie, ale jest skorelowany z `TyreLife` - warto pamietac
przy interpretacji waznosci cech.

In [ ]:
sample = laps[laps['Round'] == laps['Round'].min()]
fig = px.scatter(sample, x='LapNumber', y='LapTime_s', color='Compound',
                 title=f'Czas vs nr okrazenia - runda {int(sample.Round.iloc[0])}')
fig.show()

## Klasyfikacja: pozycja startowa vs podium

In [ ]:
rate = dr.groupby('GridPosition')['podium'].mean().reset_index()
fig = px.bar(rate, x='GridPosition', y='podium',
             title='Szansa na podium wg pozycji startowej',
             labels={'podium': 'P(podium)'})
fig.show()

In [ ]:
fig = px.histogram(dr, x='podium', color='podium',
                   title=f'Balans klas (podium rate = {dr.podium.mean():.2f}) - klasy NIEzbalansowane')
fig.show()